# 03 — Gate 1 Evaluation, UMAP & TFS Analysis (Day 11)

The held-out **test set** view of the trained dual-tower model: the Gate 1 metrics, an
interactive UMAP of the aligned 64-d embeddings, the per-cell-line Translational Fidelity
Score (TFS) ranking, a biological read on the low-fidelity outliers, and the comparison to
the random / PCA / Harmony baselines.

All numbers are read from `reports/eval_summary.json` (written by `pctrans-evaluate`) and
`data/processed/embeddings_test.npz` (written by `pctrans-visualize`) — regenerate those two
first if the model changes.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
summary = json.loads((ROOT / "reports" / "eval_summary.json").read_text())
emb = np.load(ROOT / "data" / "processed" / "embeddings_test.npz", allow_pickle=True)
print("Decision:", summary["decision"], "|", summary["decision_band"])
print("Test set:", summary["test_sizes"])

## Section 1 — Gate 1 metrics table + confusion matrix

kNN@5 retrieval accuracy is the gate metric (≥ 70% → DEPLOY). The confusion matrix rows are
the true CCLE lineage, columns the majority-vote predicted lineage from the 5 nearest TCGA
patients.

In [ ]:
kt = summary["knn_k_table"]
metrics = pd.DataFrame(
    {
        "metric": ["kNN@1", "kNN@3", "kNN@5", "kNN@10", "silhouette", "TFS (composite)"],
        "value": [kt["1"], kt["3"], kt["5"], kt["10"], summary["silhouette_score"], summary["tfs_overall"]],
    }
)
metrics.style.format({"value": "{:.3f}"})

In [ ]:
cm = pd.DataFrame(
    summary["confusion_matrix"],
    index=[f"true {x}" for x in summary["confusion_labels"]],
    columns=[f"pred {x}" for x in summary["confusion_labels"]],
)
print("Per-lineage kNN@5:", summary["per_lineage_knn"])
cm

## Section 2 — UMAP of the aligned embeddings (interactive)

UMAP is fit on the pooled CCLE + TCGA test embeddings. Colour = lineage (LUAD blue / BRCA
pink / SKCM brown); marker = domain (TCGA patient ●, CCLE cell line ✕). Hover a cross to see
the cell-line ID and its TFS.

In [ ]:
from pctrans.evaluation.viz import lineage_domain_scatter, umap_projection

z = np.concatenate([emb["z_ccle"], emb["z_tcga"]])
y = np.concatenate([emb["y_ccle"], emb["y_tcga"]])
domain = np.array([0] * len(emb["y_ccle"]) + [1] * len(emb["y_tcga"]))
ids = np.concatenate([emb["ids_ccle"], emb["ids_tcga"]])

tfs_by_id = {r["id"]: r["tfs"] for r in summary["per_cell_line_tfs"]}
tfs = np.array([tfs_by_id.get(i, np.nan) for i in ids])

coords = umap_projection(z, seed=42)
fig = lineage_domain_scatter(
    coords, y, domain, "Test-set embeddings (UMAP)", sample_ids=ids, tfs_scores=tfs
)
fig.show()

## Section 3 — TFS ranking (top / bottom 10 cell lines)

`TFS = 0.5·kNN_match + 0.5·(silhouette + 1)/2`. Per-lineage means first, then the ranked table
and the horizontal-bar figure.

In [ ]:
tfs_df = pd.DataFrame(summary["per_cell_line_tfs"])
per_lineage = (
    tfs_df.groupby("lineage")["tfs"].agg(["count", "mean", "min", "max"]).round(3)
)
per_lineage

In [ ]:
ranked = tfs_df.sort_values("tfs", ascending=False).reset_index(drop=True)
pd.concat([ranked.head(10), ranked.tail(10)])

In [ ]:
from pctrans.evaluation.viz import tfs_ranking_bar

bar = tfs_ranking_bar(tfs_df["id"].to_numpy(), tfs_df["tfs"].to_numpy(), top_n=10)
bar.show()

## Section 4 — Biological outlier analysis

The lowest-fidelity cell lines are not random noise — they are the lines whose biology sits at
the edge of their lineage. Join the bottom of the TFS ranking back to the DepMap `Model.csv`
metadata to see why.

- **ACH-000264 = Calu-6 (LUAD, TFS 0.662, silhouette 0.048)** — a classic *anaplastic /
  undifferentiated* NSCLC line derived from a **metastatic pleural effusion**. It lacks the
  clean lung-adenocarcinoma differentiation identity of the primary TCGA LUAD tumours, so it
  lands on the lineage boundary (1 of its 5 nearest TCGA patients is off-lineage). The model is
  *honest* here: low TFS flags a genuinely ambiguous line rather than hiding it.
- **Bottom BRCA lines — HCC38, HCC1937, HCC70, MDA-MB-157** — all **triple-negative / basal**
  breast lines. TCGA BRCA is dominated by ER+ luminal tumours, so the basal transcriptional
  programme pulls these lines to the near edge of the BRCA cloud (they still retrieve BRCA, but
  with the lowest silhouette in the lineage).
- **Lowest SKCM — ACH-002847 (YUHOIN)** — a **sinonasal (mucosal) melanoma** metastasis, a
  subtype distinct from the cutaneous melanoma that dominates TCGA SKCM.

In [ ]:
meta_path = ROOT / "data" / "raw" / "ccle" / "Model.csv"
cols = [
    "ModelID", "StrippedCellLineName", "OncotreeSubtype",
    "PrimaryOrMetastasis", "SampleCollectionSite",
]
if meta_path.exists():
    meta = pd.read_csv(meta_path)[cols].set_index("ModelID")
    bottom = ranked.tail(6).set_index("id").join(meta)
    display(bottom[["lineage", "tfs", "StrippedCellLineName", "OncotreeSubtype",
                    "PrimaryOrMetastasis", "SampleCollectionSite"]])
else:
    print("Model.csv not present; skipping metadata join.")

## Section 5 — Comparison to baselines

Random = 1/3 (three balanced lineages). PCA+kNN is the no-alignment baseline computed on the
same test features. Harmony is the literature reference (~63%). The contrastive dual-tower
closes the residual domain gap the unaligned baselines cannot.

In [ ]:
import plotly.graph_objects as go

b = summary["baselines"]
names = ["Random", "PCA+kNN", "Harmony (lit.)", "Ours (dual-tower)"]
vals = [b["random"], b["pca_knn"], b["harmony_reference"], summary["overall_knn_accuracy"]]
colors = ["#7f7f7f", "#7f7f7f", "#7f7f7f", "#2ca02c"]
fig = go.Figure(go.Bar(x=names, y=vals, marker_color=colors,
                       text=[f"{v * 100:.1f}%" for v in vals], textposition="outside"))
fig.add_hline(y=0.70, line_dash="dash", line_color="red",
              annotation_text="Gate 1 threshold (70%)")
fig.update_layout(title="kNN@5 retrieval accuracy vs. baselines",
                  yaxis_title="kNN@5 accuracy", yaxis_range=[0, 1.08],
                  template="plotly_white", width=700, height=450)
fig.show()